In [ ]:
import pandas as pd
import numpy as np

bureau_balance = pd.read_csv(
    "data/raw/home_credit/bureau_balance.csv",
)

bureau = pd.read_csv(
    "data/raw/home_credit/bureau.csv",
)

print("bureau_balance shape:", bureau_balance.shape)
print("bureau shape:", bureau.shape)

display(bureau_balance.head())
display(bureau_balance.describe(include="all"))

In [ ]:
bureau_balance.info()

In [ ]:
bureau_balance.isna().sum()

In [ ]:
bureau_balance.duplicated().sum()

In [ ]:
bureau_balance.duplicated(
    subset=[
        "SK_ID_BUREAU",
        "MONTHS_BALANCE"
    ]
).sum()

In [ ]:
print(
    "历史贷款数量:",
    bureau_balance["SK_ID_BUREAU"].nunique()
)

In [ ]:
# 是否属于逾期状态
bureau_balance["IS_DPD"] = (
    bureau_balance["STATUS"]
    .isin(["1", "2", "3", "4", "5"])
    .astype("int8")
)

# 是否属于严重逾期
bureau_balance["IS_SEVERE_DPD"] = (
    bureau_balance["STATUS"]
    .isin(["3", "4", "5"])
    .astype("int8")
)

# 是否为正常状态
bureau_balance["IS_STATUS_0"] = (
    bureau_balance["STATUS"] == "0"
).astype("int8")

# 是否已经结清
bureau_balance["IS_CLOSED"] = (
    bureau_balance["STATUS"] == "C"
).astype("int8")

# 是否为未知或无有效状态
bureau_balance["IS_UNKNOWN"] = (
    bureau_balance["STATUS"] == "X"
).astype("int8")

# 把数字逾期等级单独转换出来
bureau_balance["DPD_LEVEL"] = pd.to_numeric(
    bureau_balance["STATUS"],
    errors="coerce"
)

bureau_balance[
    [
        "SK_ID_BUREAU",
        "MONTHS_BALANCE",
        "STATUS",
        "IS_DPD",
        "IS_SEVERE_DPD",
        "IS_STATUS_0",
        "IS_CLOSED",
        "IS_UNKNOWN",
        "DPD_LEVEL"
    ]
].head(10)

In [ ]:
##构建月度明细特征
bureau_balance_loan_features = (
    bureau_balance
    .groupby("SK_ID_BUREAU")
    .agg(
        BB_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        BB_HISTORY_MONTHS=(
            "MONTHS_BALANCE",
            lambda x: x.max() - x.min() + 1
        ),

        BB_LATEST_MONTH=(
            "MONTHS_BALANCE",
            "max"
        ),

        BB_EARLIEST_MONTH=(
            "MONTHS_BALANCE",
            "min"
        ),

        BB_DPD_MONTH_COUNT=(
            "IS_DPD",
            "sum"
        ),

        BB_SEVERE_DPD_MONTH_COUNT=(
            "IS_SEVERE_DPD",
            "sum"
        ),

        BB_NORMAL_MONTH_COUNT=(
            "IS_STATUS_0",
            "sum"
        ),

        BB_CLOSED_MONTH_COUNT=(
            "IS_CLOSED",
            "sum"
        ),

        BB_UNKNOWN_MONTH_COUNT=(
            "IS_UNKNOWN",
            "sum"
        ),

        BB_MAX_DPD_LEVEL=(
            "DPD_LEVEL",
            "max"
        ),

        BB_AVG_DPD_LEVEL=(
            "DPD_LEVEL",
            "mean"
        )
    )
    .reset_index()
)

In [ ]:
#建立贷款级特征，包括贷款历史长度特征、贷款状态数量特征、贷款逾期程度特征、贷款状态比例特征
bureau_balance_loan_features[
    "BB_DPD_MONTH_RATIO"
] = (
    bureau_balance_loan_features[
        "BB_DPD_MONTH_COUNT"
    ]
    / bureau_balance_loan_features[
        "BB_MONTH_COUNT"
    ]
)

bureau_balance_loan_features[
    "BB_SEVERE_DPD_MONTH_RATIO"
] = (
    bureau_balance_loan_features[
        "BB_SEVERE_DPD_MONTH_COUNT"
    ]
    / bureau_balance_loan_features[
        "BB_MONTH_COUNT"
    ]
)

bureau_balance_loan_features[
    "BB_NORMAL_MONTH_RATIO"
] = (
    bureau_balance_loan_features[
        "BB_NORMAL_MONTH_COUNT"
    ]
    / bureau_balance_loan_features[
        "BB_MONTH_COUNT"
    ]
)

bureau_balance_loan_features[
    "BB_CLOSED_MONTH_RATIO"
] = (
    bureau_balance_loan_features[
        "BB_CLOSED_MONTH_COUNT"
    ]
    / bureau_balance_loan_features[
        "BB_MONTH_COUNT"
    ]
)

bureau_balance_loan_features[
    "BB_UNKNOWN_MONTH_RATIO"
] = (
    bureau_balance_loan_features[
        "BB_UNKNOWN_MONTH_COUNT"
    ]
    / bureau_balance_loan_features[
        "BB_MONTH_COUNT"
    ]
)

bureau_balance_loan_features.head()

In [ ]:
# 1. 从 bureau 中提取贷款与客户的对应关系

bureau_id_map = (
    bureau[
        [
            "SK_ID_BUREAU",
            "SK_ID_CURR"
        ]
    ]
    .drop_duplicates()
)


# 2. 将 SK_ID_CURR 合并到贷款级特征表

bureau_balance_loan_features = (
    bureau_balance_loan_features
    .merge(
        bureau_id_map,
        on="SK_ID_BUREAU",
        how="left",
        validate="one_to_one"
    )
)


# 3. 查看合并结果

bureau_balance_loan_features.head()

In [ ]:
print("bureau_id_map shape:", bureau_id_map.shape)

print(
    "重复 SK_ID_BUREAU 数量:",
    bureau_id_map["SK_ID_BUREAU"].duplicated().sum()
)

In [ ]:
print(
    "贷款级特征行数:",
    bureau_balance_loan_features.shape[0]
)

print(
    "唯一贷款数量:",
    bureau_balance_loan_features[
        "SK_ID_BUREAU"
    ].nunique()
)

In [ ]:
bureau_balance_loan_features[
    "SK_ID_CURR"
].isna().sum()

In [ ]:
#下面开始建立第四组特征:
bureau_balance_loan_features_matched = (
    bureau_balance_loan_features
    .dropna(subset=["SK_ID_CURR"])
    .copy()
)

bureau_balance_loan_features_matched[
    "SK_ID_CURR"
] = (
    bureau_balance_loan_features_matched[
        "SK_ID_CURR"
    ]
    .astype("int64")
)

In [ ]:
#删除无法匹配客户的贷款
bureau_balance_loan_features_matched = (
    bureau_balance_loan_features
    .dropna(subset=["SK_ID_CURR"])
    .copy()
)

bureau_balance_loan_features_matched[
    "SK_ID_CURR"
] = (
    bureau_balance_loan_features_matched[
        "SK_ID_CURR"
    ]
    .astype("int64")
)

In [ ]:
#检查清洗结果
print(
    "清洗前贷款数量:",
    bureau_balance_loan_features.shape[0]
)

print(
    "清洗后贷款数量:",
    bureau_balance_loan_features_matched.shape[0]
)

print(
    "SK_ID_CURR 缺失数量:",
    bureau_balance_loan_features_matched[
        "SK_ID_CURR"
    ].isna().sum()
)

In [ ]:
#进行数据清洗后，正式开始建立第四组，客户级贷款数量与历史长度特征

In [ ]:
bureau_balance_customer_base = (
    bureau_balance_loan_features_matched
    .groupby("SK_ID_CURR")
    .agg(
        BB_LOAN_COUNT=(
            "SK_ID_BUREAU",
            "count"
        ),

        BB_TOTAL_MONTH_COUNT=(
            "BB_MONTH_COUNT",
            "sum"
        ),

        BB_AVG_MONTH_COUNT=(
            "BB_MONTH_COUNT",
            "mean"
        ),

        BB_MAX_MONTH_COUNT=(
            "BB_MONTH_COUNT",
            "max"
        ),

        BB_AVG_HISTORY_MONTHS=(
            "BB_HISTORY_MONTHS",
            "mean"
        ),

        BB_MAX_HISTORY_MONTHS=(
            "BB_HISTORY_MONTHS",
            "max"
        ),

        BB_LATEST_MONTH=(
            "BB_LATEST_MONTH",
            "max"
        ),

        BB_EARLIEST_MONTH=(
            "BB_EARLIEST_MONTH",
            "min"
        )
    )
    .reset_index()
)

bureau_balance_customer_base.head()

In [ ]:
#建立客户级逾期数量特征

In [ ]:
bureau_balance_customer_dpd = (
    bureau_balance_loan_features_matched
    .groupby("SK_ID_CURR")
    .agg(
        BB_TOTAL_DPD_MONTH_COUNT=(
            "BB_DPD_MONTH_COUNT",
            "sum"
        ),

        BB_AVG_DPD_MONTH_COUNT=(
            "BB_DPD_MONTH_COUNT",
            "mean"
        ),

        BB_MAX_DPD_MONTH_COUNT=(
            "BB_DPD_MONTH_COUNT",
            "max"
        ),

        BB_TOTAL_SEVERE_DPD_MONTH_COUNT=(
            "BB_SEVERE_DPD_MONTH_COUNT",
            "sum"
        ),

        BB_AVG_SEVERE_DPD_MONTH_COUNT=(
            "BB_SEVERE_DPD_MONTH_COUNT",
            "mean"
        ),

        BB_MAX_SEVERE_DPD_MONTH_COUNT=(
            "BB_SEVERE_DPD_MONTH_COUNT",
            "max"
        )
    )
    .reset_index()
)

bureau_balance_customer_dpd.head()

In [ ]:
#建立客户级逾期程度特征

In [ ]:
bureau_balance_customer_dpd_level = (
    bureau_balance_loan_features_matched
    .groupby("SK_ID_CURR")
    .agg(
        BB_MAX_DPD_LEVEL=(
            "BB_MAX_DPD_LEVEL",
            "max"
        ),

        BB_AVG_MAX_DPD_LEVEL=(
            "BB_MAX_DPD_LEVEL",
            "mean"
        ),

        BB_AVG_DPD_LEVEL=(
            "BB_AVG_DPD_LEVEL",
            "mean"
        )
    )
    .reset_index()
)

bureau_balance_customer_dpd_level.head()

In [ ]:
#建立客户级状态比例特征

In [ ]:
bureau_balance_customer_ratio = (
    bureau_balance_loan_features_matched
    .groupby("SK_ID_CURR")
    .agg(
        BB_AVG_DPD_MONTH_RATIO=(
            "BB_DPD_MONTH_RATIO",
            "mean"
        ),

        BB_MAX_DPD_MONTH_RATIO=(
            "BB_DPD_MONTH_RATIO",
            "max"
        ),

        BB_AVG_SEVERE_DPD_MONTH_RATIO=(
            "BB_SEVERE_DPD_MONTH_RATIO",
            "mean"
        ),

        BB_MAX_SEVERE_DPD_MONTH_RATIO=(
            "BB_SEVERE_DPD_MONTH_RATIO",
            "max"
        ),

        BB_AVG_NORMAL_MONTH_RATIO=(
            "BB_NORMAL_MONTH_RATIO",
            "mean"
        ),

        BB_AVG_CLOSED_MONTH_RATIO=(
            "BB_CLOSED_MONTH_RATIO",
            "mean"
        ),

        BB_AVG_UNKNOWN_MONTH_RATIO=(
            "BB_UNKNOWN_MONTH_RATIO",
            "mean"
        )
    )
    .reset_index()
)

bureau_balance_customer_ratio.head()

In [ ]:
#将客户级特征合并

In [ ]:
bureau_balance_features = (
    bureau_balance_customer_base
    .merge(
        bureau_balance_customer_dpd,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        bureau_balance_customer_dpd_level,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        bureau_balance_customer_ratio,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

bureau_balance_features.head()

In [ ]:
#检查合并后的特征表

In [ ]:
print(
    "bureau_balance_features shape:",
    bureau_balance_features.shape
)

print(
    "唯一客户数量:",
    bureau_balance_features[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "重复客户数量:",
    bureau_balance_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
#检查缺失值
bureau_balance_features.isna().sum().sort_values(
    ascending=False
)

In [ ]:
#检查比例特征范围

In [ ]:
ratio_columns = [
    col
    for col in bureau_balance_features.columns
    if "RATIO" in col
]

bureau_balance_features[
    ratio_columns
].describe().T

In [ ]:
#检查数值分布

In [ ]:
bureau_balance_features.describe().T

In [ ]:
bureau_balance_features.to_parquet(
    "data/processed/bureau_balance_features.parquet",
    index=False
)

In [ ]:
from pathlib import Path

output_path = Path(
    "data/processed/bureau_balance_features.parquet"
)

print("文件是否存在:", output_path.exists())
print("保存位置:", output_path.resolve())
print("最终 Shape:", bureau_balance_features.shape)